# 03 – Feature engineering

Goal: turn the raw tables into one row per student, with features that could plausibly predict dropout.
Output: `data/clean/features.csv` (500 rows, one per student).

## Why each feature

| Feature | Why it should matter |
|---|---|
| `n_attempts`, `n_lessons` | How much the student engaged (EDA finding 1) |
| `mean_score`, `min_score`, `last_score` | Struggling students tend to leave (EDA finding 3); `last_score` is the most recent performance |
| `mean_time` | Minutes per quiz, a sign of effort |
| `days_since_last` | Recency of activity, measured to the course end on 30 Sep (EDA finding 2) |
| `active_span_days` | How long the student stayed active (first to last attempt) |
| `n_explanations` | Effort beyond the quiz |
| `days_since_signup` | Account age, so a new student does not look the same as an inactive one |
| `age_group`, `device`, `country` | Context; probably weak signals |

## Missing values
- Counts (`n_attempts`, `n_lessons`, `n_explanations`) are set to 0 when there is no activity: no attempts really means zero.
- The one student with no quiz attempts gets `days_since_last` = days since signup (never active since signing up) and `active_span_days` = 0.
- Scores and `mean_time` stay NaN where unknown (1 and 2 students): "unknown" is not zero. They are filled with the training-set median inside the model pipeline, so no test data leaks in.

## Not used, and why
- `refund_requested`: leakage, see the decision below.
- `student_id`: only a label, not information.
- Raw `signup_date`: replaced by `days_since_signup`.
- Explanation text and `label`: used for the LLM judge in Part 2; only the count `n_explanations` is used here.


In [1]:
#load clean data
import pandas as pd

C = "../data/clean/"
students = pd.read_csv(C + "students_clean.csv", parse_dates=["signup_date"])
quiz = pd.read_csv(C + "quiz_clean.csv", parse_dates=["attempted_at"])
expl = pd.read_csv(C + "explanations_clean.csv")
outcome = pd.read_csv(C + "outcome_clean.csv")
END = pd.Timestamp("2026-09-30 23:59:59")   # course end date

In [2]:
#per-student quiz features
g = quiz.sort_values("attempted_at").groupby("student_id")
feat = pd.DataFrame({
    "n_attempts": g.size(),
    "n_lessons": g.lesson_id.nunique(),
    "mean_score": g.score.mean(),
    "min_score": g.score.min(),
    "last_score": g.score.last(),
    "mean_time": g.time_spent_min.mean(),
    "first_attempt": g.attempted_at.min(),
    "last_attempt": g.attempted_at.max(),
})
feat["days_since_last"] = (END - feat.last_attempt).dt.days
feat["active_span_days"] = (feat.last_attempt - feat.first_attempt).dt.days
feat = feat.drop(columns=["first_attempt", "last_attempt"])
feat.head()

,n_attempts,n_lessons,mean_score,min_score,last_score,mean_time,days_since_last,active_span_days
student_id,,,,,,,,
S001,3,1,86.666667,82.0,82.0,17.000000,65,9
S002,7,3,60.571429,43.0,43.0,31.183333,38,37
S003,6,3,72.166667,57.0,57.0,22.333333,40,22
S004,3,2,61.000000,58.0,66.0,31.066667,40,19
S005,7,3,66.857143,45.0,56.0,27.685714,42,33


In [3]:
#join into one table
n_expl = expl.groupby("student_id").size().rename("n_explanations")
df = (students.merge(feat, on="student_id", how="left")
              .merge(n_expl, on="student_id", how="left")
              .merge(outcome[["student_id", "dropped"]], on="student_id"))

for c in ["n_attempts", "n_lessons", "n_explanations"]:
    df[c] = df[c].fillna(0)                    # no activity means zero

df["days_since_signup"] = (END - df.signup_date).dt.days
# the one student with no attempts has never been active: use days since signup
no_act = df.n_attempts == 0
df.loc[no_act, "days_since_last"] = df.loc[no_act, "days_since_signup"]
df.loc[no_act, "active_span_days"] = 0

print(df.shape)
df.isna().sum()

(500, 17)


student_id           0
age_group            0
country              0
signup_date          0
device               0
refund_requested     0
n_attempts           0
n_lessons            0
mean_score           1
min_score            1
last_score           1
mean_time            2
days_since_last      0
active_span_days     0
n_explanations       0
dropped              0
days_since_signup    0
dtype: int64

In [4]:
#sanity check, and your second finding table
cols = ["n_attempts", "n_lessons", "mean_score", "days_since_last",
        "active_span_days", "mean_time", "n_explanations"]
df.groupby("dropped")[cols].mean().round(2)

,n_attempts,n_lessons,mean_score,days_since_last,active_span_days,mean_time,n_explanations
dropped,,,,,,,
0,5.45,2.68,71.62,44.74,39.55,25.22,1.54
1,3.12,2.06,61.93,69.05,14.82,20.79,1.18


**What the feature table shows (dropped = 1 vs stayed = 0):**
- **Engagement is the biggest gap.** Dropouts took about 3.1 quiz attempts and covered 2.1 lessons, against 5.5 attempts and 2.7 lessons for students who stayed.
- **Dropouts went quiet sooner and stayed quiet.** Their active span is about 15 days against about 40, and their last attempt was about 69 days before the course end against about 45.
- **Scores and effort are lower, but the gap is smaller.** Mean score is about 62 vs 72, and time per quiz about 21 vs 25 minutes.
- **Explanations:** dropouts submitted fewer (1.2 vs 1.5 on average), which fits the same pattern of lower engagement.
- These are averages, not proof of cause. Part of the gap is the outcome itself: a student who left early has fewer attempts and a longer silence by 30 September (the dataset has no dropout date). The model score will be somewhat optimistic because of this.


In [5]:
#the leakage decision
pd.crosstab(df.refund_requested, df.dropped, margins=True)

dropped,0,1,All
refund_requested,,,
no,364,19,383
yes,11,106,117
All,375,125,500


**Decision: `refund_requested` is not used as a feature.**

91% of students who requested a refund dropped out (106 of 117), against 5% of those who did not (19 of 383). That is too strong to be a useful early warning. A refund request most likely happens at or after the moment a student leaves, so it would not be known when we want to predict. Using it would make the model look great in testing and be useless in practice (data leakage).

It stays in `features.csv` only so the model notebook can show how much it would inflate the score.


In [6]:
#save
df.to_csv(C + "features.csv", index=False)
print(df.columns.tolist())

['student_id', 'age_group', 'country', 'signup_date', 'device', 'refund_requested', 'n_attempts', 'n_lessons', 'mean_score', 'min_score', 'last_score', 'mean_time', 'days_since_last', 'active_span_days', 'n_explanations', 'dropped', 'days_since_signup']
